# PanDerm LoRA Fine-tuning — aptos2019 (원거리 도메인 · 5-class)

> 9차 미팅 `🧠 [학습] PEFT + Fine-tuning` · LoRA 파트

`aptos2019` (당뇨망막병증 안저영상, 5-class, train 2,562 / val 546 / test 554). 원거리 도메인 — 앞선 multi-layer 실험에서 중간층 **L11** 이 val-winner 였다. frozen 백본에 **LoRA(순수)** 와 **LoRA+융합[11,15,19,23]** 두 변형을 학습해 Linear Evaluation baseline(BACC 0.628 / Macro AUPR 0.679) 대비 성능을 본다.

공통 로직은 [`peft_ft_utils.py`](peft_ft_utils.py) 에 있다(LoRA 구현·데이터·지표·학습 루프). 이 노트북은 얇은 래퍼다.

**실행 환경**: 집의 RTX 3070(8GB) 기준. `BATCH_SIZE=8, ACCUM=4`(유효 32), AMP, grad-checkpoint. OOM 시 `BATCH_SIZE=4` 로 낮춘다.

## 0. 설정

In [1]:
import importlib
import pandas as pd
import peft_ft_utils as U
importlib.reload(U)

DATASET = "aptos2019"
cfg = U.DATASETS[DATASET]
print("device:", U.get_device())
print("checkpoint exists:", U.CHECKPOINT.exists())
print("classes:", cfg["class_names"])
# 소수 클래스: severe(3) / proliferative_dr(4)

device: cuda
checkpoint exists: True
classes: ['no_dr', 'mild', 'moderate', 'severe', 'proliferative_dr']


## 1. Baseline 정합성 확인 (frozen 마지막 CLS + linear probe)

신규 방법 수치를 신뢰하기 전에, **동일 백본·transform 으로 Linear Eval baseline 을 먼저 재현**해 문서 앵커와 ±0.02 이내인지 확인한다(기존 노트북의 alignment 관례).

In [2]:
ok, got, anchor = U.reproduce_baseline(DATASET, verbose=True)
print("baseline 재현:", "OK" if ok else "MISMATCH")
assert ok, "baseline 재현 실패 — 체크포인트/transform/split 확인 필요"

/home/junkim/miniconda3/envs/paper/lib/python3.11/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/home/junkim/miniconda3/envs/paper/lib/python3.11/site-packages/timm/models/registry.py:4: FutureWarning: Importing from timm.models.registry is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)


loading model checkpoint


/home/junkim/miniconda3/envs/paper/lib/python3.11/site-packages/torch/functional.py:534: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3595.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


VisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 1024, kernel_size=(16, 16), stride=(16, 16))
  )
  (pos_drop): Dropout(p=0.0, inplace=False)
  (blocks): ModuleList(
    (0): Block(
      (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
      (attn): Attention(
        (qkv): Linear(in_features=1024, out_features=3072, bias=False)
        (attn_drop): Dropout(p=0.0, inplace=False)
        (proj): Linear(in_features=1024, out_features=1024, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (drop_path): Identity()
      (norm2): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=1024, out_features=4096, bias=True)
        (act): GELU(approximate='none')
        (fc2): Linear(in_features=4096, out_features=1024, bias=True)
        (drop): Dropout(p=0.0, inplace=False)
      )
    )
    (1): Block(
      (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
      

100%|██████████| 9/9 [00:18<00:00,  2.07s/it]


NUM_C, Cost: 5 51.2
0
Classification report:
              precision    recall  f1-score   support

           0       0.96      0.99      0.97       272
           1       0.65      0.54      0.59        56
           2       0.70      0.81      0.75       151
           3       0.53      0.33      0.41        30
           4       0.56      0.42      0.48        45

    accuracy                           0.81       554
   macro avg       0.68      0.62      0.64       554
weighted avg       0.80      0.81      0.80       554

Debug: targets_all shape: (554,)
Debug: preds_all shape: (554,)
Debug: probs_all shape: (554, 5)
Debug: test_filenames length: 554
Model predicted results saved to /home/junkim/paper_ajou_dev/PanDerm/output_dir/_baseline_reproduce_tmp/aptos2019/aptos2019.csv
[aptos2019] reproduce BACC=0.6169 (anchor 0.6280) AUPR=0.6793 (anchor 0.6791)
baseline 재현: OK


<Figure size 1000x800 with 0 Axes>

## 2. 변형 A — 순수 LoRA + linear head (마지막 CLS)

`FUSION_LAYERS=[23]` + 전 24블록 qkv/proj LoRA + `Linear(1024→K)`. Linear Eval·Full FT 와 **깨끗한 비교축**. 학습 파라미터 <1%.

In [3]:
res_pure = U.run_lora_experiment(
    DATASET, "pure",
    lora_r=8, lora_alpha=16, batch_size=8, accum_steps=4,
    epochs=40, patience=10, use_grad_checkpoint=True, seed=0,
)
print("\n[pure] test metrics:", {k: round(v,4) for k,v in res_pure["metrics"].items() if k!="per_class_recall"})
print("[pure] bootstrap CI:", res_pure["ci"])
print("[pure] 소수 클래스 recall:", res_pure["metrics"]["per_class_recall"])

loading model checkpoint
VisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 1024, kernel_size=(16, 16), stride=(16, 16))
  )
  (pos_drop): Dropout(p=0.0, inplace=False)
  (blocks): ModuleList(
    (0): Block(
      (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
      (attn): Attention(
        (qkv): Linear(in_features=1024, out_features=3072, bias=False)
        (attn_drop): Dropout(p=0.0, inplace=False)
        (proj): Linear(in_features=1024, out_features=1024, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (drop_path): Identity()
      (norm2): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=1024, out_features=4096, bias=True)
        (act): GELU(approximate='none')
        (fc2): Linear(in_features=4096, out_features=1024, bias=True)
        (drop): Dropout(p=0.0, inplace=False)
      )
    )
    (1): Block(
      (norm1): LayerNorm((1024,), eps=1e-06, elemen

## 3. 변형 B — LoRA + 멀티레이어 융합 (ablation)

`FUSION_LAYERS=[11,15,19,23]` (중간층 결합). 융합이 성능에 기여하는지 분리 확인.

In [4]:
res_fusion = U.run_lora_experiment(
    DATASET, "fusion",
    lora_r=8, lora_alpha=16, batch_size=8, accum_steps=4,
    epochs=40, patience=10, use_grad_checkpoint=True, seed=0,
)
print("\n[fusion] test metrics:", {k: round(v,4) for k,v in res_fusion["metrics"].items() if k!="per_class_recall"})
print("[fusion] bootstrap CI:", res_fusion["ci"])
print("[fusion] 소수 클래스 recall:", res_fusion["metrics"]["per_class_recall"])

loading model checkpoint
VisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 1024, kernel_size=(16, 16), stride=(16, 16))
  )
  (pos_drop): Dropout(p=0.0, inplace=False)
  (blocks): ModuleList(
    (0): Block(
      (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
      (attn): Attention(
        (qkv): Linear(in_features=1024, out_features=3072, bias=False)
        (attn_drop): Dropout(p=0.0, inplace=False)
        (proj): Linear(in_features=1024, out_features=1024, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (drop_path): Identity()
      (norm2): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=1024, out_features=4096, bias=True)
        (act): GELU(approximate='none')
        (fc2): Linear(in_features=4096, out_features=1024, bias=True)
        (drop): Dropout(p=0.0, inplace=False)
      )
    )
    (1): Block(
      (norm1): LayerNorm((1024,), eps=1e-06, elemen

## 4. 비교표 (baseline vs LoRA-순수 vs LoRA-융합)

지표: BACC / Macro AUPR / AUROC(ovo) / W-F1 / Acc. 모두 동일 test split·동일 지표 정의.

In [5]:
base = res_pure["baseline"]
def row(name, m):
    return {"method": name, "BACC": m["bacc"], "Macro_AUPR": m["aupr"],
            "AUROC": m["auroc"], "W_F1": m["weighted_f1"], "ACC": m["acc"]}
table = pd.DataFrame([
    {"method": "Linear Eval (baseline)", "BACC": base["bacc"], "Macro_AUPR": base["aupr"],
     "AUROC": base["auroc"], "W_F1": base["weighted_f1"], "ACC": base["acc"]},
    row("LoRA (pure, last CLS)", res_pure["metrics"]),
    row("LoRA (fusion [11,15,19,23])", res_fusion["metrics"]),
]).set_index("method").round(4)
print(table)
table.to_csv(U.OUTPUT_ROOT / f"{DATASET}_lora_summary.csv")
table

                               BACC  Macro_AUPR   AUROC    W_F1     ACC
method                                                                 
Linear Eval (baseline)       0.6280      0.6791  0.8957  0.8060  0.8141
LoRA (pure, last CLS)        0.6917      0.7144  0.9051  0.7893  0.7834
LoRA (fusion [11,15,19,23])  0.6736      0.6891  0.8996  0.7894  0.7834


,BACC,Macro_AUPR,AUROC,W_F1,ACC
method,,,,,
Linear Eval (baseline),0.6280,0.6791,0.8957,0.8060,0.8141
"LoRA (pure, last CLS)",0.6917,0.7144,0.9051,0.7893,0.7834
"LoRA (fusion [11,15,19,23])",0.6736,0.6891,0.8996,0.7894,0.7834


---
**저장 위치**
- 순수: `PanDerm/output_dir/aptos2019_lora_lp/`
- 융합: `PanDerm/output_dir/aptos2019_lora_multilayer_lp/`
- 요약: `PanDerm/output_dir/aptos2019_lora_summary.csv`

각 폴더에 `training_history.csv`, `best_trainable_state_dict.pt`, `comparison_vs_baseline.csv`, `*_test_predictions.csv`.